# Step 6 — Predictor 完整 held-out evaluation

在 validation/test split 上抽取生成进度 25%、50%、75%、100% 四个位置的联合分布，评估：

- Correctness：AUROC、AUPRC、accuracy、F1、incorrect recall、Brier、ECE；
- Remaining length：MAE、median AE、预测值 vs 真值；
- reliability diagram，以及越接近结尾是否越会判断正确性。

本步骤默认评估最终 ZIP-RC 模型；模型从未用 validation/test trajectory 更新参数。

In [ ]:
# @title Step 06.1 — 初始化运行环境
from pathlib import Path
import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")
REPO_URL = "https://github.com/wtree101/ZIP-RC-Colab.git"
REPO_BRANCH = "main"
SYNC_REPO = True

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)
elif SYNC_REPO:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

if not ZIP_PY.exists():
    raise FileNotFoundError(
        f"未找到 {ZIP_PY}。请先建立 ZIP-RC 的 mamba 环境，再重新运行本 Notebook。"
    )

kernel_required = ["torch", "numpy", "pandas", "pyarrow", "matplotlib", "sklearn", "psutil"]
kernel_missing = [name for name in kernel_required if importlib.util.find_spec(name) is None]
if kernel_missing:
    raise ModuleNotFoundError(f"Colab kernel 缺少可视化依赖: {kernel_missing}")

env_check = subprocess.run(
    [
        str(ZIP_PY),
        "-c",
        (
            "import importlib.util, json; "
            "mods=['torch','vllm','transformers','datasets','pandas','pyarrow','tqdm']; "
            "print(json.dumps([m for m in mods if importlib.util.find_spec(m) is None]))"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
env_missing = json.loads(env_check.stdout.strip())
if env_missing:
    raise ModuleNotFoundError(f"zip mamba 环境缺少依赖: {env_missing}")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import torch
from IPython.display import display

sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import (
    gate,
    gate_frame,
    load_config,
    model_artifacts_exist,
    progress_frame,
    read_jsonl,
    require_columns,
    rolling_edges,
    run_repo,
    save_stage_report,
)

print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)

CONFIG = load_config(REPO)
print("Experiment:", CONFIG["experiment_name"])


In [ ]:
# @title Step 06.2 — 查看持久化进度
runtime_progress = progress_frame(REPO)
print("持久化进度快照：")
display(runtime_progress if not runtime_progress.empty else pd.DataFrame([{"状态": "尚无进度记录"}]))

In [ ]:
# @title Step 06.3 — 准备 held-out 指标
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, f1_score, recall_score, roc_auc_score

final_model = REPO / CONFIG["paths"]["final_model"]
eval_sources = {
    "validation": REPO / CONFIG["paths"]["validation"],
    "test": REPO / CONFIG["paths"]["test"],
}
positions_path = REPO / CONFIG["paths"]["predictor_positions"]
metrics_path = REPO / CONFIG["paths"]["predictor_metrics"]

def expected_calibration_error(labels, probabilities, bins=10):
    labels = np.asarray(labels, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    result = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (probabilities >= left) & (probabilities < right if right < 1 else probabilities <= right)
        if mask.any():
            result += mask.mean() * abs(labels[mask].mean() - probabilities[mask].mean())
    return float(result)

In [ ]:
# @title Step 06.4 — 运行 predictor evaluation
RUN_EVALUATION = True
if RUN_EVALUATION:
    run_repo(
        REPO,
        "python3", "src/evaluate_ziprc_predictor.py",
        "--model", final_model,
        "--data", eval_sources["validation"], eval_sources["test"],
        "--split-names", "validation", "test",
        "--out-parquet", positions_path,
        "--distribution-token-id", CONFIG["distribution_token_id"],
        "--num-length-bins", CONFIG["num_length_bins"],
        "--reward-values", *CONFIG["reward_values"],
        "--progress-points", .25, .50, .75, 1.0,
        "--max-length", CONFIG["train_max_length"],
        "--dtype", CONFIG["dtype"],
    )
position_df = pd.read_parquet(positions_path)
print("Rows:", len(position_df), "saved to", positions_path)

In [ ]:
# @title Step 06.5 — 可视化校准与长度误差
metric_rows = []
for (split_name, progress), group in position_df.groupby(["eval_split", "progress"]):
    labels = group["correct"].astype(int)
    scores = group["predicted_reward"].clip(0, 1)
    predictions = scores >= .5
    errors = (group["predicted_remaining"] - group["true_remaining"]).abs()
    naive_remaining = float(group["true_remaining"].median())
    naive_errors = (group["true_remaining"] - naive_remaining).abs()
    metric_rows.append({
        "eval_split": split_name,
        "progress": progress,
        "auroc": roc_auc_score(labels, scores) if labels.nunique() == 2 else np.nan,
        "auprc": average_precision_score(labels, scores) if labels.nunique() == 2 else np.nan,
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0),
        "incorrect_recall": recall_score(labels, predictions, pos_label=0, zero_division=0),
        "brier": brier_score_loss(labels, scores),
        "ece": expected_calibration_error(labels, scores),
        "length_mae": errors.mean(),
        "length_median_ae": errors.median(),
        "length_naive_mae": naive_errors.mean(),
    })
metrics = pd.DataFrame(metric_rows).sort_values(["eval_split", "progress"])
metrics_path.write_text(metrics.to_json(orient="records", indent=2), encoding="utf-8")
display(metrics.round(4))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
test_metrics = metrics[metrics["eval_split"] == "test"]
test_metrics.plot(x="progress", y=["auroc", "auprc"], marker="o", ax=axes[0, 0], ylim=(0, 1))
axes[0, 0].axhline(.5, color="gray", linestyle="--")
axes[0, 0].set_title("Correctness discrimination by progress")
test_metrics.plot(x="progress", y=["incorrect_recall", "f1"], marker="o", ax=axes[0, 1], ylim=(0, 1))
axes[0, 1].set_title("Error detection / F1")
test_metrics.plot(x="progress", y=["length_mae", "length_median_ae", "length_naive_mae"], marker="o", ax=axes[1, 0])
axes[1, 0].set_title("Remaining-length error")

final = position_df[(position_df["eval_split"] == "test") & (position_df["progress"] == 1.0)].copy()
final["calibration_bin"] = pd.cut(final["predicted_reward"], bins=np.linspace(0, 1, 11), include_lowest=True)
calibration = final.groupby("calibration_bin", observed=False).agg(predicted=("predicted_reward", "mean"), observed=("correct", "mean"), count=("correct", "size")).dropna()
axes[1, 1].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[1, 1].plot(calibration["predicted"], calibration["observed"], marker="o")
axes[1, 1].set(xlim=(0, 1), ylim=(0, 1), title="Reliability diagram @100%", xlabel="predicted", ylabel="observed")
plt.tight_layout()
plt.show()

final_metrics = test_metrics.iloc[-1]
early_length = test_metrics[test_metrics["progress"] < 1.0]
length_has_signal = bool((early_length["length_mae"] < early_length["length_naive_mae"]).any())
checks = [
    gate("Validation / test 均齐全", set(position_df["eval_split"]) == {"validation", "test"}, str(position_df['eval_split'].value_counts().to_dict())),
    gate("四个进度点齐全", set(position_df["progress"]) == {.25, .5, .75, 1.0}, str(sorted(position_df['progress'].unique()))),
    gate("预测值有效", position_df[["predicted_reward", "predicted_remaining"]].notna().all().all(), "no missing values"),
    gate("AUROC >0.55", final_metrics["auroc"] > .55, f"{final_metrics['auroc']:.3f}", kind="scientific"),
    gate("Incorrect recall ≥0.50", final_metrics["incorrect_recall"] >= .50, f"{final_metrics['incorrect_recall']:.3f}", kind="scientific"),
    gate("Remaining length 优于 naive", length_has_signal, "25%/50%/75% 至少一个进度点 MAE 更低", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "06_predictor_evaluation", checks, {"metrics": metrics.to_dict(orient="records")})